[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/16.%20The%20Storage%20Location%20Assignment%20Problem%20%28SLAP%29/P16-Tier-8_executed.ipynb)

---


# 16. The Storage Location Assignment Problem (SLAP)
## Tier 8: A Value-Aligned Framework for Storage Assignment

This notebook develops an illustrative value-aligned extension of a storage-location assignment problem. It combines a greedy, product-by-product assignment procedure with stakeholder-oriented scores and a separate ethical-constraint audit. The emphasis is on making the objective, assumptions, and limits of the example explicit; the numerical outputs are generated by running the code and are not specified in advance.

### Learning objectives

By the end of the notebook, you should be able to:

- distinguish a cost-only assignment baseline from a weighted value score;
- trace how the greedy procedure evaluates candidate locations and selects one for each product;
- explain how normalized scores make a weighted combination of cost efficiency and ethical indicators interpretable;
- calculate and interpret compliance metrics, penalties, and an audit summary; and
- identify which real-world effects this small example does and does not model.

### Inputs and outputs

The example generates products, locations, stakeholder descriptions, and an aggregate system state. For each product, the assignment procedure examines the available locations and selects one. The notebook reports assignment costs, value scores, constraint-audit results, and visual summaries. The generated product and location attributes are illustrative data, not observations from an operating distribution center.

### Model assumptions and scope

- Products are assigned independently, one at a time, using a greedy decision rule. A location may therefore be selected for multiple products; location capacity is recorded in the data but is not enforced.
- The cost-only baseline selects the available location with the smallest distance-times-velocity cost for each product.
- The value-aligned score combines a normalized cost-efficiency benefit with five ethical indicators. All indicators are interpreted as benefits, so larger scores are preferred.
- Ethical constraints are audited against the supplied aggregate `system_state`. The example does not derive worker workloads, service levels, energy use, or supplier counts from the assignments. Consequently, those audit metrics do not change when a candidate storage location changes, and the audit penalties are reported rather than used to select locations.
- The example does not model product-location compatibility, inventory turnover over time, travel routes, labor assignment, replenishment, or uncertainty in demand. It is an instructional framework, not a validated operational or legal compliance tool.
- No target savings, worker-satisfaction improvement, or carbon reduction is assumed. Such claims require data and a model that calculate those quantities.

### Solution sequence

1. Define stakeholder, constraint, value-function, and assignment data structures.
2. Implement constraint metrics and an auditable compliance report.
3. Construct illustrative stakeholders, products, locations, and system-state data.
4. Calculate the cost-only baseline, then make greedy value-aligned assignments.
5. Compare assignments and inspect the audit and visualization outputs.


In [ ]:
from dataclasses import dataclass
from datetime import datetime
from enum import Enum
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Representing stakeholders, constraints, and decisions

A data class groups values that describe one kind of object, while an enumeration restricts a field to a known set of categories. These structures make the inputs to the audit and assignment procedures explicit.

`ValueFunction` contains the weights used by the value-aligned score. In this example, its six weights sum to one. That convention makes them interpretable as relative importance weights, although it does not make the underlying ethical indicators interchangeable in a social or legal sense. `AssignmentDecision` records the chosen product-location pair, its cost, its component scores, and a human-readable justification.

In [ ]:
class StakeholderType(Enum):
    """Categories of parties affected by warehouse operations."""

    CUSTOMER = "customer"
    WORKER = "worker"
    MANAGEMENT = "management"
    ENVIRONMENT = "environment"
    SUPPLIER = "supplier"
    COMMUNITY = "community"


@dataclass
class Stakeholder:
    """A stakeholder's illustrative preferences, constraints, and priority."""

    stakeholder_id: str
    stakeholder_type: StakeholderType
    utility_function: Dict[str, float]
    constraints: Dict[str, Any]
    priority: float


@dataclass
class EthicalConstraint:
    """A monitored metric, threshold, penalty weight, and audit frequency."""

    constraint_id: str
    name: str
    description: str
    constraint_type: str
    metric_function: str
    threshold: float
    penalty_weight: float
    audit_frequency: str


@dataclass
class ValueFunction:
    """Relative weights for cost efficiency and ethical benefit scores."""

    cost_efficiency_weight: float = 0.4
    worker_welfare_weight: float = 0.25
    environmental_impact_weight: float = 0.15
    service_equity_weight: float = 0.1
    supplier_fairness_weight: float = 0.05
    community_impact_weight: float = 0.05


@dataclass
class AssignmentDecision:
    """One product-location choice and the information used to explain it."""

    product_id: str
    location_id: str
    cost_score: float
    ethical_scores: Dict[str, float]
    total_value_score: float
    justification: str
    stakeholder_impacts: Dict[str, float]
    timestamp: datetime

## 2. Ethical-constraint metrics and audit records

An audit metric turns an operational quantity into a number that can be compared with a threshold. For example, the worker-weight metric compares the largest recorded worker pick weight with the average. The service-equity metric uses the range of service levels, so its threshold of `0.02` means a maximum absolute spread of two percentage points when service levels are expressed on a 0-to-1 scale.

For most constraints, a metric is compliant when it is at or below its threshold. Energy reduction is different: it is a benefit, so it is compliant when the reduction ratio is at or above its target. A penalty measures the amount by which a metric misses its threshold, multiplied by a user-defined penalty weight. The penalty is an audit quantity; its units depend on that weight and it is not automatically comparable with assignment costs.

The `constraint_type` field records whether a constraint is labelled hard, soft, or preference. In this implementation, all types are evaluated and reported with the same penalty calculation; the labels do not enforce different optimization behavior.

In [ ]:
class EthicalConstraintArchitecture:
    """Evaluate aggregate ethical metrics and retain auditable records."""

    def __init__(self) -> None:
        self.constraints: List[EthicalConstraint] = []
        self.audit_history: List[Dict[str, Any]] = []
        self.violation_log: List[Dict[str, Any]] = []
        self._initialize_constraints()

    def _initialize_constraints(self) -> None:
        """Define the monitored metrics and their illustrative thresholds."""
        self.constraints = [
            EthicalConstraint(
                constraint_id="WORKER_WEIGHT_EQUITY",
                name="Worker Weight Equity",
                description="No worker handles more than 110% of average pick weight.",
                constraint_type="hard",
                metric_function="max_weight_ratio",
                threshold=1.1,
                penalty_weight=1000.0,
                audit_frequency="continuous",
            ),
            EthicalConstraint(
                constraint_id="SERVICE_EQUITY",
                name="Service Level Equity",
                description="Keep the service-level range within 2 percentage points.",
                constraint_type="hard",
                metric_function="service_level_range",
                threshold=0.02,
                penalty_weight=500.0,
                audit_frequency="daily",
            ),
            EthicalConstraint(
                constraint_id="ENERGY_REDUCTION",
                name="Energy Consumption Reduction",
                description="Reduce energy consumption by at least 15% from baseline.",
                constraint_type="soft",
                metric_function="energy_reduction_ratio",
                threshold=0.15,
                penalty_weight=200.0,
                audit_frequency="weekly",
            ),
            EthicalConstraint(
                constraint_id="SUPPLIER_NEUTRALITY",
                name="Supplier Treatment Neutrality",
                description="Monitor relative variation in supplier assignment counts.",
                constraint_type="soft",
                metric_function="supplier_assignment_variation",
                threshold=0.1,
                penalty_weight=100.0,
                audit_frequency="weekly",
            ),
            EthicalConstraint(
                constraint_id="COMMUNITY_NOISE",
                name="Community Noise Impact",
                description="Limit the share of locations classified as high-noise.",
                constraint_type="preference",
                metric_function="noise_impact_score",
                threshold=0.3,
                penalty_weight=50.0,
                audit_frequency="continuous",
            ),
            EthicalConstraint(
                constraint_id="WORKLOAD_DISTRIBUTION",
                name="Fair Workload Distribution",
                description="Monitor squared relative variation in worker workloads.",
                constraint_type="hard",
                metric_function="workload_variation",
                threshold=0.15,
                penalty_weight=300.0,
                audit_frequency="daily",
            ),
        ]

    def evaluate_constraint_compliance(
        self,
        assignment_decisions: List[AssignmentDecision],
        system_state: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Evaluate every constraint and append one audit record."""
        del assignment_decisions  # Metrics in this example use aggregate system data.
        compliance_results: Dict[str, Dict[str, Any]] = {}
        violations: List[Dict[str, Any]] = []
        total_penalty = 0.0

        for constraint in self.constraints:
            metric_value = self._calculate_metric(
                constraint.metric_function, system_state
            )
            is_energy_target = (
                constraint.metric_function == "energy_reduction_ratio"
            )
            if is_energy_target:
                compliant = metric_value >= constraint.threshold
                shortfall = max(0.0, constraint.threshold - metric_value)
            else:
                compliant = metric_value <= constraint.threshold
                shortfall = max(0.0, metric_value - constraint.threshold)

            penalty = shortfall * constraint.penalty_weight
            total_penalty += penalty
            compliance_results[constraint.constraint_id] = {
                "name": constraint.name,
                "metric_value": metric_value,
                "threshold": constraint.threshold,
                "compliant": compliant,
                "penalty": penalty,
                "severity": (
                    "high" if penalty > 500 else
                    "medium" if penalty > 100 else
                    "low"
                ),
            }

            if not compliant:
                violations.append(
                    {
                        "constraint_id": constraint.constraint_id,
                        "name": constraint.name,
                        "metric_value": metric_value,
                        "threshold": constraint.threshold,
                        "penalty": penalty,
                        "timestamp": datetime.now(),
                    }
                )

        compliance_rate = (
            sum(result["compliant"] for result in compliance_results.values())
            / len(compliance_results)
            if compliance_results
            else 1.0
        )
        audit_record = {
            "timestamp": datetime.now(),
            "total_penalty": total_penalty,
            "compliance_rate": compliance_rate,
            "violations_count": len(violations),
            "constraint_results": compliance_results,
            "violations": violations,
        }
        self.audit_history.append(audit_record)
        self.violation_log.extend(violations)

        return {
            "total_penalty": total_penalty,
            "compliance_rate": compliance_rate,
            "violations": violations,
            "constraint_results": compliance_results,
        }

    def _calculate_metric(
        self, metric_function: str, system_state: Dict[str, Any]
    ) -> float:
        """Calculate a named metric from the supplied aggregate state."""
        if metric_function == "max_weight_ratio":
            values = system_state.get("worker_weights", [])
            if not values:
                return 0.0
            average = float(np.mean(values))
            return float(max(values) / average) if average > 0 else 0.0

        if metric_function == "service_level_range":
            values = system_state.get("service_levels", [])
            return float(max(values) - min(values)) if len(values) >= 2 else 0.0

        if metric_function == "energy_reduction_ratio":
            current = float(system_state.get("current_energy_consumption", 0.0))
            baseline = float(system_state.get("baseline_energy_consumption", 0.0))
            if baseline <= 0:
                raise ValueError("Baseline energy consumption must be positive.")
            return (baseline - current) / baseline

        if metric_function == "supplier_assignment_variation":
            counts = list(system_state.get("supplier_assignments", {}).values())
            if len(counts) < 2:
                return 0.0
            average = float(np.mean(counts))
            return float(np.var(counts) / average**2) if average > 0 else 0.0

        if metric_function == "noise_impact_score":
            total = float(system_state.get("total_locations", 0))
            high_noise = float(system_state.get("high_noise_locations", 0))
            if total <= 0:
                raise ValueError("Total locations must be positive.")
            return high_noise / total

        if metric_function == "workload_variation":
            values = system_state.get("worker_workloads", [])
            if len(values) < 2:
                return 0.0
            average = float(np.mean(values))
            return float(np.var(values) / average**2) if average > 0 else 0.0

        raise ValueError(f"Unknown metric function: {metric_function}")

    def generate_transparency_report(self) -> Dict[str, Any]:
        """Summarize the most recent audits and their constraint results."""
        if not self.audit_history:
            return {"status": "no_audit_data"}

        recent_audits = self.audit_history[-10:]
        constraint_compliance: Dict[str, Dict[str, Any]] = {}
        for constraint in self.constraints:
            outcomes = [
                audit["constraint_results"][constraint.constraint_id]["compliant"]
                for audit in recent_audits
                if constraint.constraint_id in audit["constraint_results"]
            ]
            if outcomes:
                rate = float(np.mean(outcomes))
                status = (
                    "good" if rate >= 0.95 else
                    "needs_attention" if rate >= 0.8 else
                    "critical"
                )
                constraint_compliance[constraint.constraint_id] = {
                    "name": constraint.name,
                    "compliance_rate": rate,
                    "type": constraint.constraint_type,
                    "status": status,
                }

        violations = [
            violation
            for audit in recent_audits
            for violation in audit["violations"]
        ]
        return {
            "report_timestamp": datetime.now(),
            "audit_period": f"Last {len(recent_audits)} audits",
            "overall_compliance": float(
                np.mean([audit["compliance_rate"] for audit in recent_audits])
            ),
            "average_penalty": float(
                np.mean([audit["total_penalty"] for audit in recent_audits])
            ),
            "total_violations": sum(
                audit["violations_count"] for audit in recent_audits
            ),
            "violations": violations,
            "constraint_compliance": constraint_compliance,
            "recommendations": self._generate_recommendations(
                constraint_compliance
            ),
        }

    def _generate_recommendations(
        self, constraint_compliance: Dict[str, Any]
    ) -> List[str]:
        """Turn repeated low compliance into concise monitoring guidance."""
        recommendations = []
        for result in constraint_compliance.values():
            if result["status"] == "critical":
                recommendations.append(
                    f"Review recurring compliance issues for {result['name']}."
                )
            elif result["status"] == "needs_attention":
                recommendations.append(
                    f"Monitor compliance for {result['name']} closely."
                )
        if not recommendations:
            recommendations.append(
                "No recurring low-compliance pattern appears in the audited records."
            )
        return recommendations

## 3. Greedy value-aligned assignment

A greedy procedure makes a sequence of local choices: it considers one product, evaluates the available locations for that product, selects the best-scoring location, and then moves to the next product. It is straightforward to trace, but it does not guarantee a globally optimal assignment when decisions interact. Here, the independence assumption means that each product is evaluated separately and location capacity is not enforced.

For product $i$ and candidate location $j$, let the cost be

$$
c_{ij} = d_j v_i,
$$

where $d_j$ is the location distance and $v_i$ is the product velocity. The baseline chooses a location with the smallest $c_{ij}$ for each product.

The value-aligned procedure converts cost into a benefit on the interval $[0,1]$. For a product with candidate costs $c_{ij}$, define

$$
e_{ij} =
\begin{cases}
1 - \dfrac{c_{ij} - c_i^{\min}}{c_i^{\max} - c_i^{\min}}, & c_i^{\max} > c_i^{\min}, \\
1, & c_i^{\max} = c_i^{\min},
\end{cases}
$$

where $c_i^{\min}$ and $c_i^{\max}$ are the smallest and largest candidate costs for product $i$. Thus, the least costly candidate receives cost-efficiency benefit 1 and the most costly receives 0; if all candidate costs are equal, every candidate receives 1.

Let $s_{ij}^{(k)} \in [0,1]$ be the score for ethical dimension $k$ and let $w_k$ be its weight. The procedure maximizes

$$
V_{ij} = w_c e_{ij} + \sum_{k \in K} w_k s_{ij}^{(k)},
\qquad
j_i^* \in \operatorname*{arg\,max}_{j \in L} V_{ij}.
$$

Here, $w_c$ is the cost-efficiency weight, $K$ is the set of five ethical dimensions, and $L$ is the set of candidate locations. The implementation uses the fields in `ValueFunction`, `_calculate_ethical_scores`, and `_perform_ethical_optimization`. Ties are resolved by the order of `locations`, because the code replaces the current choice only when it finds a strictly larger score. The audit penalties are kept separate: the audited system metrics do not depend on the candidate location in this example.

If there are $n$ products and $m$ locations, evaluating every candidate in this product-by-product procedure takes $O(nm)$ score evaluations, aside from the cost of calculating the scores. This is not a joint capacity-constrained assignment model.

In [ ]:
class ValueAlignedOptimizer:
    """Compare a greedy cost baseline with a weighted-score assignment."""

    def __init__(
        self,
        value_function: ValueFunction,
        ethical_architecture: EthicalConstraintArchitecture,
    ) -> None:
        self.value_function = value_function
        self.ethical_architecture = ethical_architecture
        self.optimization_history: List[Dict[str, Any]] = []
        self.decision_log: List[AssignmentDecision] = []

    def optimize_assignment(
        self,
        products: List[Dict[str, Any]],
        locations: List[Dict[str, Any]],
        stakeholders: List[Stakeholder],
        system_state: Dict[str, Any],
    ) -> List[AssignmentDecision]:
        """Run the baseline, greedy score selection, and aggregate audits."""
        if not products:
            raise ValueError("At least one product is required.")
        if not locations:
            raise ValueError("At least one location is required.")

        print("Running value-aligned SLAP assignment...")
        baseline_assignments = self._calculate_baseline_assignments(
            products, locations
        )
        baseline_compliance = (
            self.ethical_architecture.evaluate_constraint_compliance(
                baseline_assignments, system_state
            )
        )
        ethical_assignments = self._perform_ethical_optimization(
            products,
            locations,
            stakeholders,
            system_state,
            baseline_assignments,
        )
        ethical_compliance = (
            self.ethical_architecture.evaluate_constraint_compliance(
                ethical_assignments, system_state
            )
        )

        for decision in ethical_assignments:
            decision.justification = self._generate_decision_justification(
                decision, stakeholders
            )

        baseline_cost = sum(item.cost_score for item in baseline_assignments)
        ethical_cost = sum(item.cost_score for item in ethical_assignments)
        optimization_record = {
            "timestamp": datetime.now(),
            "baseline_assignments": baseline_assignments,
            "ethical_assignments": ethical_assignments,
            "baseline_cost": baseline_cost,
            "ethical_cost": ethical_cost,
            "baseline_compliance": baseline_compliance["compliance_rate"],
            "ethical_compliance": ethical_compliance["compliance_rate"],
            "baseline_penalty": baseline_compliance["total_penalty"],
            "ethical_penalty": ethical_compliance["total_penalty"],
            "baseline_compliance_results": baseline_compliance[
                "constraint_results"
            ],
            "ethical_compliance_results": ethical_compliance[
                "constraint_results"
            ],
            "value_summary": self._calculate_value_improvement(
                baseline_assignments, ethical_assignments
            ),
        }
        self.optimization_history.append(optimization_record)
        self.decision_log.extend(ethical_assignments)

        print(
            f"Baseline audit compliance: "
            f"{baseline_compliance['compliance_rate']:.1%}"
        )
        print(
            f"Value-aligned audit compliance: "
            f"{ethical_compliance['compliance_rate']:.1%}"
        )
        return ethical_assignments

    def _calculate_baseline_assignments(
        self,
        products: List[Dict[str, Any]],
        locations: List[Dict[str, Any]],
    ) -> List[AssignmentDecision]:
        """Choose the least-cost location independently for each product."""
        assignments = []
        for product in products:
            location = min(
                locations,
                key=lambda candidate: (
                    candidate["distance"] * product["velocity"]
                ),
            )
            cost = location["distance"] * product["velocity"]
            assignments.append(
                AssignmentDecision(
                    product_id=product["id"],
                    location_id=location["id"],
                    cost_score=cost,
                    ethical_scores={},
                    total_value_score=0.0,
                    justification="Lowest-cost candidate for this product.",
                    stakeholder_impacts={},
                    timestamp=datetime.now(),
                )
            )
        return assignments

    def _perform_ethical_optimization(
        self,
        products: List[Dict[str, Any]],
        locations: List[Dict[str, Any]],
        stakeholders: List[Stakeholder],
        system_state: Dict[str, Any],
        baseline_assignments: List[AssignmentDecision],
    ) -> List[AssignmentDecision]:
        """Select each product's highest-scoring candidate location."""
        del system_state, baseline_assignments  # Audits are aggregate and separate.
        assignments = []

        for product in products:
            candidate_data = []
            for location in locations:
                cost = location["distance"] * product["velocity"]
                scores = self._calculate_ethical_scores(
                    product, location, stakeholders, {}
                )
                candidate_data.append((location, cost, scores))

            costs = [item[1] for item in candidate_data]
            minimum_cost = min(costs)
            maximum_cost = max(costs)
            best_decision = None
            best_value_score = -float("inf")

            for location, cost, ethical_scores in candidate_data:
                if maximum_cost > minimum_cost:
                    cost_benefit = 1.0 - (
                        (cost - minimum_cost) / (maximum_cost - minimum_cost)
                    )
                else:
                    cost_benefit = 1.0

                value_score = (
                    self.value_function.cost_efficiency_weight * cost_benefit
                    + self.value_function.worker_welfare_weight
                    * ethical_scores["worker_welfare"]
                    + self.value_function.environmental_impact_weight
                    * ethical_scores["environmental_impact"]
                    + self.value_function.service_equity_weight
                    * ethical_scores["service_equity"]
                    + self.value_function.supplier_fairness_weight
                    * ethical_scores["supplier_fairness"]
                    + self.value_function.community_impact_weight
                    * ethical_scores["community_impact"]
                )

                if value_score > best_value_score:
                    best_value_score = value_score
                    best_decision = AssignmentDecision(
                        product_id=product["id"],
                        location_id=location["id"],
                        cost_score=cost,
                        ethical_scores=ethical_scores,
                        total_value_score=value_score,
                        justification="",
                        stakeholder_impacts={},
                        timestamp=datetime.now(),
                    )

            if best_decision is not None:
                assignments.append(best_decision)
        return assignments

    def _calculate_ethical_scores(
        self,
        product: Dict[str, Any],
        location: Dict[str, Any],
        stakeholders: List[Stakeholder],
        system_state: Dict[str, Any],
    ) -> Dict[str, float]:
        """Return illustrative benefit scores bounded between zero and one."""
        del stakeholders, system_state  # These scores use the listed product/location fields.
        distance = float(location["distance"])
        ergonomic_score = float(location.get("ergonomic_score", 0.0))
        if ergonomic_score > 0:
            worker_welfare = ergonomic_score
        else:
            worker_welfare = 1.0 - distance / 100.0

        energy_efficiency = float(location.get("energy_efficiency", 0.0))
        if energy_efficiency > 0:
            environmental_impact = energy_efficiency
        else:
            environmental_impact = 1.0 - distance / 50.0

        segment = product.get("customer_segment", "standard")
        service_equity = {
            "premium": 0.8,
            "standard": 0.9,
            "economy": 1.0,
        }.get(segment, 0.9)

        supplier_rank = float(product.get("supplier_rank", 5))
        supplier_fairness = 1.0 - abs(supplier_rank - 5.5) / 4.5
        community_impact = (
            0.3 if location.get("near_residential", False) else 0.8
        )

        return {
            "worker_welfare": float(np.clip(worker_welfare, 0.0, 1.0)),
            "environmental_impact": float(
                np.clip(environmental_impact, 0.0, 1.0)
            ),
            "service_equity": float(np.clip(service_equity, 0.0, 1.0)),
            "supplier_fairness": float(
                np.clip(supplier_fairness, 0.0, 1.0)
            ),
            "community_impact": float(np.clip(community_impact, 0.0, 1.0)),
        }

    def _generate_decision_justification(
        self,
        decision: AssignmentDecision,
        stakeholders: List[Stakeholder],
    ) -> str:
        """Summarize the recorded cost and high-scoring dimensions."""
        del stakeholders
        reasons = [f"Assignment cost score: {decision.cost_score:.2f}"]
        labels = {
            "worker_welfare": "strong worker-welfare score",
            "environmental_impact": "strong environmental score",
            "service_equity": "strong service-equity score",
            "supplier_fairness": "strong supplier-fairness score",
            "community_impact": "strong community-impact score",
        }
        for key, label in labels.items():
            if decision.ethical_scores.get(key, 0.0) > 0.7:
                reasons.append(label)
        return "; ".join(reasons)

    def _calculate_value_improvement(
        self,
        baseline_assignments: List[AssignmentDecision],
        ethical_assignments: List[AssignmentDecision],
    ) -> Dict[str, float]:
        """Report cost change and ethical-score average without an assumed baseline."""
        baseline_cost = sum(item.cost_score for item in baseline_assignments)
        ethical_cost = sum(item.cost_score for item in ethical_assignments)
        cost_change_percentage = (
            (ethical_cost - baseline_cost) / baseline_cost * 100.0
            if baseline_cost != 0
            else 0.0
        )
        ethical_score_average = float(
            np.mean(
                [
                    np.mean(list(item.ethical_scores.values()))
                    for item in ethical_assignments
                    if item.ethical_scores
                ]
            )
        ) if ethical_assignments else 0.0
        return {
            "cost_change_percentage": cost_change_percentage,
            "ethical_score_average": ethical_score_average,
        }

    def generate_optimization_report(self) -> Dict[str, Any]:
        """Return the latest assignment summary and audit report."""
        if not self.optimization_history:
            return {"status": "no_optimization_data"}
        latest = self.optimization_history[-1]
        return {
            "latest_optimization": latest,
            "total_optimizations": len(self.optimization_history),
            "total_decisions": len(self.decision_log),
            "transparency_report": (
                self.ethical_architecture.generate_transparency_report()
            ),
        }

## 4. Create the illustrative distribution-center data

The next cell constructs six stakeholder profiles, 100 products, 50 locations, and aggregate system-state measurements. Product and location attributes are sampled when the cell runs; no random seed is set, so repeated runs can produce different assignments. The stakeholder priorities and the weights in `ValueFunction` are descriptive inputs; the priority values are not used as multipliers in the assignment score.

The sample values for worker weights, service levels, energy, supplier counts, noise, and workloads are audit inputs. They are not inferred from the generated products or locations.

In [ ]:
def create_value_aligned_example() -> Tuple[
    List[Stakeholder], List[Dict[str, Any]], List[Dict[str, Any]], Dict[str, Any]
]:
    """Create illustrative stakeholders, products, locations, and system data."""
    stakeholders = [
        Stakeholder(
            "STAKEHOLDER_CUSTOMER", StakeholderType.CUSTOMER,
            {"service_level": 0.6, "price": 0.4},
            {"min_service_level": 0.95}, 0.30,
        ),
        Stakeholder(
            "STAKEHOLDER_WORKER", StakeholderType.WORKER,
            {"safety": 0.4, "workload_balance": 0.3, "wages": 0.3},
            {"max_weight_ratio": 1.1, "max_work_hours": 8}, 0.25,
        ),
        Stakeholder(
            "STAKEHOLDER_MANAGEMENT", StakeholderType.MANAGEMENT,
            {"profitability": 0.5, "efficiency": 0.3, "growth": 0.2},
            {"min_roi": 0.15, "max_labor_cost": 1000000}, 0.20,
        ),
        Stakeholder(
            "STAKEHOLDER_ENVIRONMENT", StakeholderType.ENVIRONMENT,
            {"energy_efficiency": 0.5, "carbon_footprint": 0.5},
            {"max_energy_consumption": 1000000, "min_carbon_reduction": 0.15},
            0.15,
        ),
        Stakeholder(
            "STAKEHOLDER_SUPPLIER", StakeholderType.SUPPLIER,
            {"fair_treatment": 0.6, "payment_terms": 0.4},
            {"min_assignment_fairness": 0.8}, 0.05,
        ),
        Stakeholder(
            "STAKEHOLDER_COMMUNITY", StakeholderType.COMMUNITY,
            {"noise_reduction": 0.4, "local_employment": 0.3, "safety": 0.3},
            {"max_noise_level": 65, "min_local_jobs": 100}, 0.05,
        ),
    ]

    product_categories = [
        "electronics", "clothing", "home_goods", "groceries", "pharma"
    ]
    customer_segments = ["premium", "standard", "economy"]
    products = []
    for index in range(100):
        products.append(
            {
                "id": f"PRODUCT_{index:03d}",
                "velocity": float(np.random.uniform(10, 100)),
                "size": float(np.random.uniform(0.5, 5.0)),
                "category": str(np.random.choice(product_categories)),
                "customer_segment": str(np.random.choice(customer_segments)),
                "supplier_rank": int(np.random.randint(1, 11)),
                "weight": float(np.random.uniform(1, 50)),
            }
        )

    location_types = ["premium", "standard", "bulk", "eco_friendly"]
    locations = []
    for index in range(50):
        locations.append(
            {
                "id": f"LOCATION_{index:02d}",
                "distance": float(np.random.uniform(5, 50)),
                "capacity": float(np.random.uniform(10, 100)),
                "type": str(np.random.choice(location_types)),
                "ergonomic_score": float(np.random.uniform(0.3, 1.0)),
                "energy_efficiency": float(np.random.uniform(0.4, 1.0)),
                "near_residential": bool(np.random.random() < 0.2),
            }
        )

    system_state = {
        "worker_weights": [45, 42, 48, 50, 38, 44, 46, 43],
        "service_levels": [0.96, 0.94, 0.95, 0.97, 0.93],
        "current_energy_consumption": 850000,
        "baseline_energy_consumption": 1000000,
        "supplier_assignments": {
            "SUPPLIER_A": 25,
            "SUPPLIER_B": 30,
            "SUPPLIER_C": 22,
            "SUPPLIER_D": 23,
        },
        "high_noise_locations": 8,
        "total_locations": 50,
        "worker_workloads": [7.5, 8.2, 7.8, 8.0, 7.3, 7.9, 8.1, 7.6],
    }
    return stakeholders, products, locations, system_state


value_function = ValueFunction(
    cost_efficiency_weight=0.4,
    worker_welfare_weight=0.25,
    environmental_impact_weight=0.15,
    service_equity_weight=0.1,
    supplier_fairness_weight=0.05,
    community_impact_weight=0.05,
)
ethical_architecture = EthicalConstraintArchitecture()
optimizer = ValueAlignedOptimizer(value_function, ethical_architecture)
stakeholders, products, locations, system_state = create_value_aligned_example()

weight_total = sum(vars(value_function).values())
assert np.isclose(weight_total, 1.0), "Value-function weights must sum to one."
assert len({product["id"] for product in products}) == len(products)
assert len({location["id"] for location in locations}) == len(locations)

print("Illustrative example created.")
print(f"Stakeholders: {len(stakeholders)}")
print(f"Products: {len(products)}")
print(f"Locations: {len(locations)}")
print(f"Audited constraints: {len(ethical_architecture.constraints)}")

## 5. Run the assignments and inspect the decisions

The next cell runs the cost-only baseline and then the weighted-score procedure. The reported cost difference is the baseline cost minus the value-aligned cost: a positive value means the value-aligned choices have lower total assignment cost, and a negative value means they have higher total assignment cost. It is an outcome of this generated instance, not a guaranteed property of the method.

The compliance rates summarize the aggregate system-state audit. Because those inputs do not depend on the selected storage locations, equal baseline and value-aligned audit rates are expected under the stated model assumptions; the audit is not evidence that the assignment procedure caused a change in worker welfare, energy use, or service equity.

In [ ]:
ethical_assignments = optimizer.optimize_assignment(
    products, locations, stakeholders, system_state
)
latest = optimizer.optimization_history[-1]
baseline_assignments = latest["baseline_assignments"]
baseline_cost = latest["baseline_cost"]
ethical_cost = latest["ethical_cost"]
cost_difference = baseline_cost - ethical_cost
cost_difference_percentage = (
    cost_difference / baseline_cost * 100.0 if baseline_cost != 0 else 0.0
)

assert len(ethical_assignments) == len(products)
assert len(baseline_assignments) == len(products)
assert np.isclose(ethical_cost, sum(item.cost_score for item in ethical_assignments))
assert np.isclose(
    baseline_cost, sum(item.cost_score for item in baseline_assignments)
)

print("Assignment comparison")
print(f"Products assigned: {len(ethical_assignments)}")
print(f"Cost-only baseline total: {baseline_cost:,.2f}")
print(f"Value-aligned total: {ethical_cost:,.2f}")
print(
    f"Baseline minus value-aligned cost: {cost_difference:,.2f} "
    f"({cost_difference_percentage:.2f}% of baseline)"
)
print(
    "Audit compliance rates: "
    f"baseline={latest['baseline_compliance']:.1%}, "
    f"value-aligned={latest['ethical_compliance']:.1%}"
)

print("\nSample value-aligned decisions")
for index, decision in enumerate(ethical_assignments[:5], start=1):
    print(
        f"{index}. {decision.product_id} -> {decision.location_id}; "
        f"cost={decision.cost_score:.2f}; "
        f"value score={decision.total_value_score:.3f}"
    )
    print(f"   Rationale: {decision.justification}")

print("\nStakeholder profiles")
for stakeholder in stakeholders:
    print(
        f"{stakeholder.stakeholder_type.value.title()}: "
        f"priority={stakeholder.priority:.2f}; "
        f"profile constraints={list(stakeholder.constraints)}"
    )

## 6. Visualize assignment and audit summaries

The panels show the two total assignment costs, the aggregate compliance rates, compliance frequency by constraint across the audits performed, stakeholder priority inputs, the distribution of selected ethical scores, and the value-function weights. These plots summarize the example; they do not establish causal effects or operational performance beyond the quantities explicitly modeled.

In [ ]:
def visualize_value_aligned_results(
    optimizer: ValueAlignedOptimizer,
    ethical_architecture: EthicalConstraintArchitecture,
    stakeholders: List[Stakeholder],
) -> None:
    """Plot assignment, score, stakeholder-input, and audit summaries."""
    if not optimizer.optimization_history:
        raise ValueError("Run the assignment procedure before plotting results.")

    latest = optimizer.optimization_history[-1]
    report = ethical_architecture.generate_transparency_report()
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle("Value-Aligned SLAP: Assignment and Audit Summary", fontsize=15)

    cost_ax = axes[0, 0]
    cost_labels = ["Cost-only baseline", "Value-aligned"]
    costs = [latest["baseline_cost"], latest["ethical_cost"]]
    bars = cost_ax.bar(cost_labels, costs, color=["#4C78A8", "#F58518"])
    cost_ax.set_title("Total Assignment Cost")
    cost_ax.set_ylabel("Cost (distance × velocity units)")
    cost_ax.tick_params(axis="x", rotation=15)
    cost_ax.grid(axis="y", alpha=0.3)
    for bar, value in zip(bars, costs):
        cost_ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:,.1f}",
            ha="center",
            va="bottom",
        )

    compliance_ax = axes[0, 1]
    rates = [latest["baseline_compliance"], latest["ethical_compliance"]]
    bars = compliance_ax.bar(
        ["Baseline audit", "Value-aligned audit"],
        rates,
        color=["#72B7B2", "#54A24B"],
    )
    compliance_ax.set_title("Aggregate Audit Compliance")
    compliance_ax.set_ylabel("Share of compliant constraints")
    compliance_ax.set_ylim(0, 1.05)
    compliance_ax.tick_params(axis="x", rotation=15)
    compliance_ax.grid(axis="y", alpha=0.3)
    for bar, rate in zip(bars, rates):
        compliance_ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{rate:.1%}",
            ha="center",
        )

    constraint_ax = axes[0, 2]
    constraint_data = report.get("constraint_compliance", {})
    if constraint_data:
        names = [item["name"] for item in constraint_data.values()]
        constraint_rates = [
            item["compliance_rate"] for item in constraint_data.values()
        ]
        constraint_colors = [
            "#54A24B" if rate >= 0.95 else
            "#F2CF5B" if rate >= 0.8 else
            "#E45756"
            for rate in constraint_rates
        ]
        constraint_ax.barh(names, constraint_rates, color=constraint_colors)
        constraint_ax.set_xlim(0, 1)
        constraint_ax.set_xlabel("Compliance frequency across recent audits")
        constraint_ax.set_title("Constraint Audit History")
        constraint_ax.grid(axis="x", alpha=0.3)
    else:
        constraint_ax.text(0.5, 0.5, "No audit records", ha="center")
        constraint_ax.set_axis_off()

    priority_ax = axes[1, 0]
    priority_labels = [item.stakeholder_type.value.title() for item in stakeholders]
    priorities = [item.priority for item in stakeholders]
    priority_ax.barh(priority_labels, priorities, color="#B279A2")
    priority_ax.set_title("Stakeholder Priority Inputs")
    priority_ax.set_xlabel("Declared priority (not used in score calculation)")
    priority_ax.grid(axis="x", alpha=0.3)

    score_ax = axes[1, 1]
    ethical_values = [
        score
        for decision in optimizer.optimization_history[-1]["ethical_assignments"]
        for score in decision.ethical_scores.values()
    ]
    if ethical_values:
        score_ax.hist(
            ethical_values,
            bins=20,
            range=(0, 1),
            color="#72B7B2",
            edgecolor="white",
        )
        score_ax.axvline(
            float(np.mean(ethical_values)),
            color="#E45756",
            linestyle="--",
            label=f"Mean: {np.mean(ethical_values):.3f}",
        )
        score_ax.legend()
    score_ax.set_title("Selected Ethical Indicator Scores")
    score_ax.set_xlabel("Score (0 = lower benefit, 1 = higher benefit)")
    score_ax.set_ylabel("Frequency")
    score_ax.grid(axis="y", alpha=0.3)

    weight_ax = axes[1, 2]
    weight_labels = [
        "Cost efficiency",
        "Worker welfare",
        "Environmental impact",
        "Service equity",
        "Supplier fairness",
        "Community impact",
    ]
    weights = list(vars(optimizer.value_function).values())
    weight_ax.barh(weight_labels, weights, color="#4C78A8")
    weight_ax.set_title("Value-Function Weights")
    weight_ax.set_xlabel("Weight")
    weight_ax.grid(axis="x", alpha=0.3)

    fig.tight_layout()
    plt.show()

    print("Audit summary")
    print(f"Audit period: {report['audit_period']}")
    print(f"Overall compliance: {report['overall_compliance']:.1%}")
    print(f"Average audit penalty: {report['average_penalty']:.2f}")
    print(f"Recorded violations: {report['total_violations']}")
    for recommendation in report["recommendations"]:
        print(f"- {recommendation}")


visualize_value_aligned_results(
    optimizer, ethical_architecture, stakeholders
)

## 7. Inspect constraint metrics and stakeholder links

This final analysis displays the latest aggregate audit, groups recorded violations by constraint type, and maps selected constraints to stakeholder groups for interpretation. The stakeholder scores below are a simple summary of audited constraint outcomes, not measured satisfaction. A compliant constraint contributes 1 and a non-compliant constraint contributes 0.5; stakeholders with no mapped constraints are reported as not assessed.

The governance summary describes audit coverage and recorded compliance. It does not establish that the assignment algorithm detects bias, provides legal accountability, or measures stakeholder trust.

In [ ]:
def analyze_ethical_constraints(
    ethical_architecture: EthicalConstraintArchitecture,
    optimizer: ValueAlignedOptimizer,
) -> None:
    """Display the latest audit, mapped stakeholder indicators, and limitations."""
    if not optimizer.optimization_history:
        raise ValueError("Run the assignment procedure before auditing results.")

    latest = optimizer.optimization_history[-1]
    results = latest["ethical_compliance_results"]
    print("Constraint performance in the latest aggregate audit")
    print(
        f"{'Constraint':<32} {'Type':<12} {'Metric':>10} "
        f"{'Threshold':>10} {'Status':>12} {'Penalty':>10}"
    )
    print("-" * 92)
    for constraint in ethical_architecture.constraints:
        result = results[constraint.constraint_id]
        status = "Compliant" if result["compliant"] else "Not compliant"
        print(
            f"{constraint.name:<32} {constraint.constraint_type:<12} "
            f"{result['metric_value']:>10.4f} "
            f"{result['threshold']:>10.4f} {status:>12} "
            f"{result['penalty']:>10.2f}"
        )

    violations_by_type: Dict[str, List[Dict[str, Any]]] = {}
    for violation in latest["ethical_compliance_results"].values():
        del violation
    latest_violations = [
        violation
        for audit in ethical_architecture.audit_history[-2:]
        for violation in audit["violations"]
    ]
    for violation in latest_violations:
        constraint = next(
            item
            for item in ethical_architecture.constraints
            if item.constraint_id == violation["constraint_id"]
        )
        violations_by_type.setdefault(constraint.constraint_type, []).append(
            violation
        )

    print("\nRecorded violations in the two latest audits")
    if not violations_by_type:
        print("No violations were recorded in those audit records.")
    else:
        for constraint_type, violations in violations_by_type.items():
            print(f"{constraint_type.title()} constraints: {len(violations)}")
            for violation in violations:
                print(
                    f"  {violation['name']}: metric="
                    f"{violation['metric_value']:.4f}, "
                    f"threshold={violation['threshold']:.4f}, "
                    f"penalty={violation['penalty']:.2f}"
                )

    constraint_stakeholder_map = {
        "SERVICE_EQUITY": "Customer",
        "WORKER_WEIGHT_EQUITY": "Worker",
        "WORKLOAD_DISTRIBUTION": "Worker",
        "ENERGY_REDUCTION": "Environment",
        "SUPPLIER_NEUTRALITY": "Supplier",
        "COMMUNITY_NOISE": "Community",
    }
    stakeholder_outcomes: Dict[str, List[float]] = {}
    for constraint_id, stakeholder_name in constraint_stakeholder_map.items():
        if constraint_id in results:
            score = 1.0 if results[constraint_id]["compliant"] else 0.5
            stakeholder_outcomes.setdefault(stakeholder_name, []).append(score)

    all_stakeholders = [
        "Customer", "Worker", "Management", "Environment", "Supplier", "Community"
    ]
    print("\nConstraint-based stakeholder indicators")
    for name in all_stakeholders:
        values = stakeholder_outcomes.get(name, [])
        if not values:
            print(f"{name:<15} Not assessed")
            continue
        score = float(np.mean(values))
        label = "Higher" if score >= 0.9 else "Mixed" if score >= 0.7 else "Review"
        print(f"{name:<15} indicator={score:.1%}; category={label}")

    print("\nInterpretation")
    print(
        "The baseline and value-aligned audits use the same aggregate system state. "
        "Their compliance metrics therefore describe that state, not an effect "
        "caused by a particular storage assignment."
    )
    print(
        "Constraint labels and audit frequencies are metadata in this example; "
        "the code does not implement separate legal or enforcement procedures."
    )


analyze_ethical_constraints(ethical_architecture, optimizer)

## 8. Interpretation and appropriate use

The notebook demonstrates how to make stakeholder-oriented indicators visible in a storage-assignment workflow while retaining a cost-only baseline. The greedy value-aligned rule trades off normalized cost efficiency against five bounded scores according to the declared weights. The audit evaluates separate aggregate metrics and records threshold shortfalls as penalties.

The distinction between assignment scoring and system auditing is important: a score can influence a location choice only if it varies with that choice, while an audit can describe system conditions without being an optimization constraint. In this example, the location-level scores influence the greedy choice, whereas the aggregate audit metrics do not. The stakeholder profiles and priorities provide context but do not directly enter the assignment formula.

Before using a framework like this operationally, decision-makers would need to define and validate the meaning, units, and data sources for each score and threshold; determine how stakeholder priorities are governed; model capacity and product-location feasibility; and connect assignment decisions to the operational outcomes being audited. Those additions would extend the model beyond the instructional scope of this notebook.